# DT401 Workshop 1 — SQL Part 4: Combining Data with JOINs

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the difference between `LEFT JOIN` and `INNER JOIN`
- Combine data from multiple tables using join conditions
- Chain multiple joins together to traverse relationships across several tables
- Use table aliases to simplify queries and avoid ambiguous column names
- Apply joins to answer real-world questions spanning multiple datasets

---

## Context Recap

In the previous notebook, you learned to sort and aggregate data within a single table. Real-world databases rarely store everything in one place — customer addresses sit in an `address` table, city names in a `city` table, and so on. JOINs let you combine these separate tables into a single result, answering questions that span multiple datasets: "Which customers live in Canada?" or "How many rentals happened in each city?"

**Prerequisites:** Complete [3 — Sorting and Aggregating](3_sorting_and_aggregating.ipynb) before starting this notebook.

Run this cell before starting. If your kernel restarts, run it again.

In [ ]:
# Setup: load the SQL extension and connect to the pagila database
%load_ext sql
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False
%config SqlMagic.autopolars = False
%config SqlMagic.displaycon = False

%sql sqlite:///data/pagila.db

---

## How JOINs Work

A JOIN combines rows from two tables based on a matching column — typically a primary key in one table and a foreign key in the other. The general syntax is:

```sql
SELECT t1.*, t2.column
FROM table1 t1
LEFT JOIN table2 t2 ON t1.shared_key = t2.shared_key;
```

The two most common join types are:

| Type | Returns |
|---|---|
| `LEFT JOIN` | All rows from the left table; matching rows from the right table. Where there is no match, the right-table columns contain NULL. |
| `INNER JOIN` | Only rows where a match exists in **both** tables. Rows without a match are dropped entirely. |

### Seeing the Difference in Practice

Consider the `customer` and `address` tables. Every customer has an `address_id`, but imagine a customer record was loaded without a corresponding address row. Run the same join both ways and the difference becomes clear:

```sql
-- LEFT JOIN: returns all 599 customers; address columns are NULL for any without a match
SELECT c.customer_id, c.first_name, a.address
FROM customer c
LEFT JOIN address a ON c.address_id = a.address_id;

-- INNER JOIN: returns only customers who have a matching address row
SELECT c.customer_id, c.first_name, a.address
FROM customer c
INNER JOIN address a ON c.address_id = a.address_id;
```

In the pagila database both queries return the same count because every customer has an address. In real-world data this is rarely the case — records go missing, imports fail, and referential integrity is not always enforced. **Use LEFT JOIN when you want to keep all records from your main table, even if some don't have matches in the joined table. Use INNER JOIN when you only want records that exist in both tables** — for example, when finding customers and actors who share a first name (Q83).

<div class="alert alert-info">

**Table aliases**

Table aliases (e.g. `customer c`) save typing and are essential when joining tables that share column names. Once you define an alias, reference columns as `c.first_name` instead of `customer.first_name`. When joining multiple tables with common column names — such as `customer_id` appearing in both `customer` and `payment` — aliases prevent ambiguous column errors.

</div>

---

## Section 6: Combining Tables with JOINs

Customer information in the pagila database is spread across four tables. The `customer` table holds names and a reference to an address; the `address`, `city`, and `country` tables hold the geographic detail. The ERD below shows how they connect:

```mermaid
erDiagram
    customer ||--o{ address : "address_id"
    address ||--|{ city : "city_id"
    city ||--|{ country : "country_id"

    customer {
        integer customer_id PK
        varchar first_name
        varchar last_name
        smallint address_id
    }
    address {
        integer address_id PK
        varchar address
        smallint city_id
    }
    city {
        integer city_id PK
        varchar city
        smallint country_id
    }
    country {
        integer country_id PK
        varchar country
    }
```

The `address_id` column appears in both `customer` and `address` — this is the link that lets you join them. Each join in the chain follows the same principle: match on the shared key between the current table and the next one.

Questions 77–79 build this chain one table at a time. Start with a single join, verify the result, then extend it. This incremental approach — add one join, check the output, then add the next — is a reliable strategy when working with unfamiliar data.

#### Question 77

> Join the `address` column from the `address` table onto the `customer` table.

In [ ]:
%%sql


<details>
<summary>Solution to Question 77</summary>

```sql
%%sql
SELECT
    c.*,
    a.address
FROM customer c
LEFT JOIN address a ON c.address_id = a.address_id;
```

The result adds a single `address` column to every customer row. Because every customer in pagila has a matching address, no NULL values appear — but in real datasets you would often see NULLs here, indicating customers whose address records are missing or incomplete.

</details>

#### Question 78

> Extend your previous query to also join the `city` column onto the `customer` table. This will require two joins.

In [ ]:
%%sql


<details>
<summary>Solution to Question 78</summary>

```sql
%%sql
SELECT
    cu.*,
    ad.address,
    ct.city
FROM customer cu
LEFT JOIN address ad ON cu.address_id = ad.address_id
LEFT JOIN city ct ON ad.city_id = ct.city_id;
```

The second join connects `address` to `city` via `city_id`. Notice that the second join is from `address` (the previously joined table) rather than from `customer` — each new join extends the chain by one link. Using distinct aliases (`cu`, `ad`, `ct`) avoids any ambiguity between the tables.

</details>

#### Question 79

> Extend further to also join `country` onto the customer table. This will require three joins.

In [ ]:
%%sql


<details>
<summary>Solution to Question 79</summary>

```sql
%%sql
SELECT
    c.*,
    a.address,
    c1.city,
    c2.country
FROM customer c
LEFT JOIN address a ON c.address_id = a.address_id
LEFT JOIN city c1 ON a.city_id = c1.city_id
LEFT JOIN country c2 ON c1.country_id = c2.country_id;
```

With three joins you now have a full address — street, city, and country — alongside each customer's name and ID. Note that `c1` and `c2` are used to distinguish `city` and `country`, which both start with the same letter. Choosing clear, consistent aliases becomes especially important as the number of joins grows.

</details>

Films, actors, and languages are linked through a different part of the ERD. The `film_actor` table is a **junction table** — it has no meaningful data of its own, but exists solely to record the many-to-many relationship between films and actors (one film has many actors; one actor appears in many films).

```mermaid
erDiagram
    actor ||--o{ film_actor : "actor_id"
    film ||--|{ film_actor : "film_id"
    film ||--o{ inventory : "film_id"
    language ||--|{ film : "language_id"

    actor {
        integer actor_id PK
        varchar first_name
        varchar last_name
    }
    film {
        integer film_id PK
        varchar title
        smallint language_id
        mpaa_rating rating
        smallint length
    }
    film_actor {
        smallint actor_id
        smallint film_id
    }
    language {
        integer language_id PK
        character name
    }
```

Questions 80–83 explore this part of the schema. Q83 uses INNER JOIN in a context where it is the right choice — finding names that appear in *both* tables.

#### Question 80

> Join the film `title` onto the `inventory` table.

In [ ]:
%%sql


<details>
<summary>Solution to Question 80</summary>

```sql
%%sql
SELECT
    i.*,
    f.title
FROM inventory i
LEFT JOIN film f ON i.film_id = f.film_id;
```

The `inventory` table records physical copies of films. On its own, it shows only `film_id` — not particularly readable. Joining `film` replaces the numeric ID with a title, which is the pattern you will use constantly when working with normalised databases: IDs are stored for efficiency, but names are joined in when you need to present or analyse the data.

</details>

#### Question 81

> Join the film language (the `name` column from the `language` table) onto the `inventory` table. You will need two joins — through `film` first.

In [ ]:
%%sql


<details>
<summary>Solution to Question 81</summary>

```sql
%%sql
SELECT
    i.*,
    f.title AS film_title,
    l.name AS language
FROM inventory i
LEFT JOIN film f ON i.film_id = f.film_id
LEFT JOIN language l ON f.language_id = l.language_id;
```

There is no direct link between `inventory` and `language` — you must pass through `film` to get there. This is the same chaining logic as Q78 and Q79. Notice the use of `AS` to rename `name` to `language`, which is more descriptive in the output and avoids potential confusion with the reserved word `name`.

</details>

#### Question 82

> Join together all films and actors so that actors appear next to the films they starred in. Use `film_actor` as your base table.

In [ ]:
%%sql


<details>
<summary>Solution to Question 82</summary>

```sql
%%sql
SELECT
    f.title AS film_title,
    a.first_name,
    a.last_name
FROM film_actor fa
LEFT JOIN film f ON fa.film_id = f.film_id
LEFT JOIN actor a ON fa.actor_id = a.actor_id;
```

Starting from `film_actor` and joining outwards in both directions is the standard approach for a junction table. The result has one row per film-actor combination — so a film with 10 actors appears 10 times, once for each actor. This is exactly the structure you need for counting actors per film or films per actor.

</details>

#### Question 83

> Find all `first_name` values in the `customer` table that are also present in the `actor` table. Use an `INNER JOIN` to find the matches.

In [ ]:
%%sql


<details>
<summary>Solution to Question 83</summary>

```sql
%%sql
SELECT DISTINCT cu.first_name
FROM customer cu
INNER JOIN actor ac ON cu.first_name = ac.first_name
ORDER BY cu.first_name;
```

This is a different use of JOIN from the previous questions. Here you are not joining on a numeric key — you are joining on a text value to find overlap between two lists. INNER JOIN is the right choice because you only want names that exist in *both* tables; customers with no actor match (or actors with no customer match) should be excluded entirely. LEFT JOIN would have returned all customers, with NULL in the actor column for those without a match — which is not what the question asks for.

</details>

---

## Section 7: Using JOINs with Other Keywords

JOINs become most powerful when combined with the filtering, grouping, and sorting keywords from earlier notebooks. The pattern is always the same: join first to bring the relevant columns into scope, then filter, group, and sort on those combined columns.

The ERD below shows the payment chain — the tables you will need for the remaining questions:

```mermaid
erDiagram
    customer ||--o{ payment : "customer_id"
    rental ||--o{ payment : "rental_id"
    inventory ||--o{ rental : "inventory_id"
    film ||--|{ inventory : "film_id"

    customer {
        integer customer_id PK
        varchar first_name
        varchar last_name
        smallint store_id
    }
    payment {
        integer payment_id PK
        smallint customer_id
        numeric amount
    }
```

Questions 84–86 increase in complexity: Q84 groups by customer and orders the result, Q85 adds a WHERE filter to that query, and Q86 replaces ORDER BY with HAVING to filter on the aggregate itself. This is the same progression you practised at the end of Notebook 3, now applied across joined tables.

#### Question 84

> Select the first name, last name, and total sales amount for each customer. What is the name of the customer who spent the most?

In [ ]:
%%sql


<details>
<summary>Solution to Question 84</summary>

```sql
%%sql
SELECT
    c.first_name,
    c.last_name,
    SUM(p.amount) AS total_sales
FROM payment p
LEFT JOIN customer c ON p.customer_id = c.customer_id
GROUP BY c.first_name, c.last_name
ORDER BY SUM(amount) DESC;
```

Karl Seal spent the most. Note that `payment` is the base table here rather than `customer` — this is deliberate. Starting from `payment` ensures that every payment row is included in the sum. If you started from `customer` and joined payments, customers with no payments would appear with NULL amounts rather than being excluded, and you would need to handle those NULLs separately.

</details>

#### Question 85

> Select first name, last name, and total sales for customers from store 1 only. Who spent the most from store 1?

In [ ]:
%%sql


<details>
<summary>Solution to Question 85</summary>

```sql
%%sql
SELECT
    c.first_name,
    c.last_name,
    SUM(p.amount) AS total_sales
FROM payment p
LEFT JOIN customer c ON p.customer_id = c.customer_id
WHERE store_id = 1
GROUP BY c.first_name, c.last_name
ORDER BY SUM(amount) DESC;
```

Eleanor Hunt spent the most in store 1. The WHERE clause filters rows before grouping, so only store 1 customers' payments are included in the sums. The `store_id` column comes from the `customer` table, which is available here because it was joined in — you can filter on any column that is in scope, not just columns from the base table.

</details>

#### Question 86

> What are the names of the customers who have spent more than £190 in total?

In [ ]:
%%sql


<details>
<summary>Solution to Question 86</summary>

```sql
%%sql
SELECT
    c.first_name,
    c.last_name,
    SUM(amount) AS total_sales
FROM payment p
LEFT JOIN customer c ON p.customer_id = c.customer_id
GROUP BY c.customer_id
HAVING SUM(amount) > 190;
```

Five customers spent more than £190 in total, including Eleanor Hunt and Karl Seal. HAVING is necessary here because you are filtering on an aggregate value (`SUM(amount)`). As covered in Notebook 3, WHERE runs before grouping and cannot reference aggregates — HAVING runs after grouping, which is exactly when the total is available to compare against.

</details>

#### Question 87

> Find the average payment amount for customers from each city. Return two columns: city name and average amount. This will require 4 tables and 3 joins.

**Hint:** Start with the `payment` table, then work outward one step at a time: `payment` → `customer` → `address` → `city`. Each join links to the next table using the shared key visible in the ERD above and the Section 6 ERD.

In [ ]:
%%sql


<details>
<summary>Solution to Question 87</summary>

```sql
%%sql
SELECT
    ct.city,
    AVG(p.amount) AS avg_payment
FROM payment p
LEFT JOIN customer cu ON p.customer_id = cu.customer_id
LEFT JOIN address ad ON cu.address_id = ad.address_id
LEFT JOIN city ct ON ad.city_id = ct.city_id
GROUP BY ct.city;
```

This query chains the same four tables you joined in Q79 — but this time you are aggregating by city rather than listing individual customers. Building the join chain is the same process in both cases; what changes is the SELECT and GROUP BY clauses at the top and bottom. Once you can chain joins confidently, the analytical possibilities multiply: you can group, filter, and aggregate across any combination of the joined columns.

</details>

Questions 88 and 89 return to the film-actor portion of the schema from Section 6. Both questions combine JOINs with WHERE and GROUP BY to answer real analytical questions about the data.

#### Question 88

> What is the name of the actor who appears in the most PG-rated films?

In [ ]:
%%sql


<details>
<summary>Solution to Question 88</summary>

```sql
%%sql
SELECT
    a.first_name,
    a.last_name,
    COUNT(f.film_id) AS num_pg_films
FROM film_actor fa
LEFT JOIN film f ON fa.film_id = f.film_id
LEFT JOIN actor a ON fa.actor_id = a.actor_id
WHERE f.rating = 'PG'
GROUP BY a.actor_id, a.first_name, a.last_name
ORDER BY num_pg_films DESC;
```

Cameron Zellweger appears in 15 PG-rated films. The WHERE clause filters for PG films before grouping, so the count reflects only PG appearances. Grouping by `a.actor_id` as well as the name columns ensures that two actors with the same name are counted separately — a good habit to form even when duplicates seem unlikely.

</details>

#### Question 89

> What is the name of the actor who appears in the longest films, on average?

In [ ]:
%%sql


<details>
<summary>Solution to Question 89</summary>

```sql
%%sql
SELECT
    a.first_name,
    a.last_name,
    AVG(f.length) AS avg_length
FROM film_actor fa
LEFT JOIN film f ON fa.film_id = f.film_id
LEFT JOIN actor a ON fa.actor_id = a.actor_id
GROUP BY a.actor_id, a.first_name, a.last_name
ORDER BY avg_length DESC;
```

Judy Dean appears in films with an average length of 138 minutes. This query uses the same join structure as Q88 but removes the WHERE filter (all ratings are included) and replaces COUNT with AVG on the `length` column. It is a useful reminder that once the join is in place, switching between analytical questions is often just a matter of changing the aggregate function and the column it operates on.

</details>

---

## Summary

In this notebook you have covered:

| Keyword / Function | Purpose |
|---|---|
| `LEFT JOIN table ON condition` | Join keeping all left-table rows; NULLs where no match in right table |
| `INNER JOIN table ON condition` | Join keeping only rows matched in both tables |
| `table_name alias` | Short name for a table, used to qualify column references |
| `col AS alias` | Rename a column in the output |

JOINs can be chained (two, three, or more in a single query) and combined with all previous keywords — WHERE, GROUP BY, ORDER BY, and HAVING. The general strategy for a complex multi-table query is: identify the tables you need, trace the join path through the ERD, build the joins one at a time, and then layer in filtering and aggregation.

**Next:** [5 — Subqueries and Creating Tables](5_subqueries_and_tables.ipynb)